### Create control table

In [0]:
from landreg.config import OPS_SCHEMA, table

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {table(OPS_SCHEMA, "control")} (
        source_name STRING NOT NULL,
        load_type STRING NOT NULL,
        source_path STRING NOT NULL,
        target_table STRING NOT NULL,
        is_enabled BOOLEAN NOT NULL
    )
""")

### Populate control table

In [0]:
spark.sql(f"""
    MERGE INTO {table(OPS_SCHEMA, "control")} AS target
    USING (
        SELECT * FROM VALUES
            ('price_paid_complete', 'batch', 'price-paid/complete/', 'price_paid_complete', true),
            ('price_paid_monthly', 'stream', 'price-paid/monthly/', 'price_paid_monthly', true)
        AS source(source_name, load_type, source_path, target_table, is_enabled)
    ) AS source
    ON target.source_name = source.source_name
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

### Create run log table

In [0]:
from landreg.config import OPS_SCHEMA, table

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {table(OPS_SCHEMA, "run_log")} (
        run_id STRING NOT NULL,
        source_name STRING NOT NULL,
        started_at TIMESTAMP NOT NULL,
        status STRING NOT NULL,
        ended_at TIMESTAMP,
        rows_written BIGINT,
        error_message STRING
    )
""")